# Momepy roads - Viladecans

This notebook uses momepy to calculate geometrical and topological features of the Viladecans street network:
- **Street profile**: `width`, `openness`, `width_deviation` (requires OSM buildings)
- **Betweenness centrality** (node-based, local radius 500 m, propagated to edges via mean_nodes)
- **Closeness centrality** -- extended local (radius 800 m) and local 400 m
- **Straightness centrality** (node-based, radius 500 m)

Input: `VIL_noise_streets.gpkg` -- the Viladecans OSM drive streets with acoustic zone noise classes.

Local-radius variants are used for tractability, matching the Berlin pipeline.

## Import libraries

In [ ]:
import os
import momepy as mm
import geopandas as gpd
import osmnx as ox
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

## Import streets segments

In [ ]:
_layers_dir = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'layers')
edges = gpd.read_file(os.path.join(_layers_dir, 'VIL_noise_streets.gpkg'))
print(edges.crs)
print(edges.shape)
print(edges.columns.tolist())
print('Number of street segments:', len(edges))

In [ ]:
place_name = 'Viladecans, Spain'
tags = {'building': True}

print('Downloading buildings from OSM...')
try:
    buildings = ox.features_from_place(place_name, tags)
except AttributeError:
    buildings = ox.geometries_from_place(place_name, tags)

print(f'Downloaded {len(buildings)} buildings.')
buildings = buildings[buildings.geometry.type.isin(['Polygon', 'MultiPolygon'])]
buildings = buildings.to_crs(edges.crs)
print(f'Polygon buildings: {len(buildings)}')

## Create Momepy street profile

In [ ]:
profile = mm.street_profile(edges, buildings)
profile.head()

In [ ]:
edges[profile.columns] = profile
edges[profile.columns].head()

In [ ]:
ax = edges.plot(cmap='plasma', column='width', linewidth=0.5, legend=True, figsize=(12, 12))
ax.set_title('Total Street Width')
ax.set_axis_off()

In [ ]:
ax = edges.plot(cmap='plasma', column='openness', linewidth=0.5, legend=True, figsize=(12, 12))
ax.set_title('Openness')
ax.set_axis_off()

# Centrality

Build the primal graph once; all centrality functions operate on it.

In momepy 0.11 all centralities are node-based. `mm.mean_nodes()` propagates each
node attribute to its incident edges (mean of the two endpoint values) before the
edge GeoDataFrame is extracted.

In [ ]:
primal = mm.gdf_to_nx(edges, approach="primal")
print('Graph nodes:', primal.number_of_nodes(), '  edges:', primal.number_of_edges())

## Betweenness Centrality

In [ ]:
betweenness = mm.betweenness_centrality(
    primal, name='betweenness_metric_e', mode='edges', weight='mm_len',
)

In [ ]:
primal_gdf_betw = mm.nx_to_gdf(betweenness, points=False)
primal_gdf_betw.head()

In [ ]:
ax = primal_gdf_betw.plot(
    column='betweenness_metric_e',
    cmap='Spectral_r',
    scheme='quantiles',
    alpha=0.8,
    legend=True,
    figsize=(12, 12)
)
ax.set_axis_off()
ax.set_title('Betweenness Centrality - Edge Mean (r=500 m)')

## Global Closeness Centrality

Careful, takes a few minutes

In [ ]:
closeness = mm.closeness_centrality(
    primal, name='closeness_global', weight='mm_len'
)

In [ ]:
# Propagate node closeness to edges and rebuild the edge GeoDataFrame
mm.mean_nodes(closeness, 'closeness_global')
primal_gdf_clos = mm.nx_to_gdf(closeness, points=False)
primal_gdf_clos.head()

In [ ]:
ax = primal_gdf_clos.plot(
    column='closeness_global',
    cmap='Spectral_r',
    scheme='quantiles',
    k=15,
    alpha=0.6,
    figsize=(12, 12),
    legend=True
)
ax.set_axis_off()
_ = ax.set_title('Closeness Global Edge Mean (r=800 m)')

## Local Closeness Centrality (radius 400 m)

In [ ]:
local_clos = mm.closeness_centrality(
    primal, radius=400, name='closeness400', distance='mm_len', weight='mm_len'
)

In [ ]:
mm.mean_nodes(local_clos, 'closeness400')
local_clos_gdf = mm.nx_to_gdf(local_clos, points=False)

ax = local_clos_gdf.plot(
    column='closeness400',
    cmap='Spectral_r',
    scheme='quantiles',
    k=15,
    alpha=0.6,
    figsize=(15, 15),
)
ax.set_axis_off()
_ = ax.set_title('closeness400')

## Straightness (node-based, radius 500 m)

In [ ]:
straighteness = mm.straightness_centrality(
    primal, radius=500, distance='mm_len'
)

In [ ]:
mm.mean_nodes(straighteness, 'straightness')
primal_gdf_straight = mm.nx_to_gdf(straighteness, points=False)
primal_gdf_straight.head()

In [ ]:
ax = primal_gdf_straight.plot(
    column='straightness',
    cmap='Spectral_r',
    scheme='quantiles',
    k=15,
    alpha=0.6,
    figsize=(12, 12),
)
ax.set_axis_off()
_ = ax.set_title('Straightness (r=500 m)')

## Merge all centrality features

Combine width/openness (from street profile), betweenness, closeness and straightness
into a single edge GeoDataFrame keyed by `segment_id`.

In [ ]:
# primal_gdf_straight carries width, openness and other profile columns because
# mm.gdf_to_nx preserves all edge attributes from the input GDF
primal_gdf = primal_gdf_straight.copy()

def _safe_merge(base, other, col):
    if 'segment_id' in other.columns and 'segment_id' in base.columns:
        return base.merge(other[['segment_id', col]], on='segment_id', how='left')
    base = base.copy()
    base[col] = other[col].values
    return base

primal_gdf = _safe_merge(primal_gdf, primal_gdf_betw, 'betweenness_metric_e')
primal_gdf = _safe_merge(primal_gdf, primal_gdf_clos, 'closeness_global')
primal_gdf = _safe_merge(primal_gdf, local_clos_gdf, 'closeness400')

print(primal_gdf.columns.tolist())
primal_gdf.head()

## Create data frame

Column names match the Berlin `momepy_roads_features.csv` schema consumed by
`01_VIL_create_dataset_class.ipynb`.

In [ ]:
def _col(gdf, col):
    """Return column if present, else a zeros Series."""
    return gdf[col] if col in gdf.columns else pd.Series(0.0, index=gdf.index)

dataset = pd.DataFrame({
    'road_id':          _col(primal_gdf, 'segment_id'),
    'width':            _col(primal_gdf, 'width'),
    'openness':         _col(primal_gdf, 'openness'),
    'height':           _col(primal_gdf, 'height'),
    'betweenness':      _col(primal_gdf, 'betweenness_metric_e'),
    'closeness_global': _col(primal_gdf, 'closeness_global'),
    'closeness400':     _col(primal_gdf, 'closeness400'),
    'straightness':     _col(primal_gdf, 'straightness'),
}).fillna(0)

dataset = dataset.drop_duplicates(subset='road_id', keep='first')
dataset.head()

## Export dataset to CSV

In [ ]:
output_dir = 'data'
os.makedirs(output_dir, exist_ok=True)
dataset.to_csv(os.path.join(output_dir, 'momepy_roads_features.csv'), index=False)
print('Exported momepy_roads_features.csv')